# Swap Pricing and Risk

Price a synthetic market SOFR swap under collateral-aware discounting and review scenario diagnostics.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.bootstrap import build_full_curves, load_market_data
from src.instruments import build_periods
from src.risk import key_rate_dv01, run_scenarios

market = load_market_data(PROJECT_ROOT)
result = build_full_curves(PROJECT_ROOT)
quote = market.swaps[-1]
periods = build_periods(quote.start_date, quote.end_date, quote.pay_freq, quote.day_count, calendar=market.config.market.calendar, roll=market.config.market.business_day_roll)
scenarios = run_scenarios(100_000_000, quote.fixed_rate, periods, result.discount_curve, result.projection_curve)
scenarios

In [ ]:
krd = key_rate_dv01(100_000_000, quote.fixed_rate, periods, result.discount_curve, result.projection_curve, list(market.config.risk.key_rates_years))
plt.figure(figsize=(9, 4))
plt.bar(scenarios['Scenario'], scenarios['Swap PV'])
plt.xticks(rotation=45, ha='right')
plt.title('Scenario PV for 10Y Synthetic SOFR Swap')
plt.tight_layout()
plt.show()
krd